# 01 — Audio Basics
## Controllable Sound Generation Playground
**Amaç:** WAV → waveform → spectrogram → mel-spectrogram dönüşümünü görmek ve terimleri öğrenmek.
Bu notebook bir model eğitmez; ses temsillerini inceleyen küçük bir deneydir. Aday kontrol parametrelerini henüz seçmiyoruz.

**Çalıştırma:** Dosyayı repository içindeki `notebooks/` klasörüne koy. VS Code'da Python 3.11 `.venv` kernel'ini seç ve **Run All** kullan. Kurulu paketler: numpy, matplotlib, librosa, soundfile ve jupyter. Kernel yanlışsa import hataları alabilirsin.

Kendi WAV dosyan varsa aşağıda `AUDIO_PATH` değerini değiştir. Yoksa notebook üç kısa darbe içeren sentetik bir demo WAV oluşturur; internet veya dataset gerekmez. Bu ses gerçek Foley kaydı değildir.

## 1. Temel sözlük
| Terim | Anlamı |
|---|---|
| WAV | Ses verisini ve örnekleme bilgilerini taşıyan dosya biçimi. WAV bir grafik değildir; her WAV aynı kodlamayı kullanmaz. Burada PCM kullanıyoruz. |
| Sample / örnek | Dijital sesin bir zaman noktasındaki genlik değeri. |
| Sampling rate / örnekleme hızı (`sr`) | Saniyedeki örnek sayısı; birimi Hz. 24 000 Hz, saniyede 24 000 örnek demektir. |
| Amplitude / genlik | Sinyalin anlık değeri. Float PCM analizinde genellikle yaklaşık −1 ile +1 aralığında çalışılır. Fiziksel ses basıncıyla aynı değildir. |
| Frequency / frekans | Saniyedeki titreşim sayısı, Hz. Algılanan pitch ile ilişkili ama aynı kavram değil. |
| Mono / stereo | Bir / iki ses kanalı. Kanal ortalaması stereo bilgiyi kaybettirir; karşı fazdaki kanallar birbirini söndürebilir. |
| Bit depth | PCM örneğinin hassasiyeti: örneğin 16 bit. Örnekleme hızıyla farklıdır. |
| Duration / süre | `örnek sayısı / sr`, saniye. |
| Nyquist frekansı | `sr / 2`. Örnekleme öncesi bu sınırın üzerindeki içerik filtrelenmelidir; aksi halde aliasing oluşabilir. |
| Clipping | Genliğin temsil sınırında kesilmesi ve bozulması. Yalnızca tepe değeri ölçümü clipping'in kesin kanıtı değildir. |

**Önemli:** Dosyanın `sr` değerini değiştirmek, sesi yeniden örneklemek değildir. Gerçek resampling yeni örnekler ve gerekli filtreleme ile yapılır. Bu notebook dosyanın doğal `sr` değerini korur.

In [ ]:
from pathlib import Path
import sys
import json
import csv
import platform
import hashlib
import importlib.metadata
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, display

print("Python:", sys.version.split()[0])
print("librosa:", librosa.__version__)
plt.rcParams.update({"figure.figsize": (11, 3), "axes.grid": False})
# Repo kökünden veya notebooks/ klasöründen çalıştırılabilir.
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
AUDIO_PATH = None  # Örnek: ROOT / "data" / "raw" / "impact.wav"

# Her yeniden çalıştırma aynı deney dosyalarını günceller.
FIGURES = ROOT / "results" / "figures" / "01_audio_basics"
TABLES = ROOT / "results" / "tables" / "01_audio_basics"
EXPERIMENT = ROOT / "experiments" / "01_audio_basics"
for folder in [FIGURES, TABLES, EXPERIMENT]:
    folder.mkdir(parents=True, exist_ok=True)


## 2. WAV oluştur veya kendi dosyanı seç
Demo içinde 0.4, 1.2 ve 2.0 saniyede kısa darbeler var. **Onset** başlangıç anı, **decay** başlangıçtan sonra genliğin sönmesidir. Sinüs bileşenleri ve deterministik gürültü yalnızca anlaşılır bir örnek üretmek için kullanılıyor.

In [ ]:
if AUDIO_PATH is None:
    demo_sr = 24000
    t = np.arange(3 * demo_sr) / demo_sr
    demo = np.zeros_like(t)
    rng = np.random.default_rng(42)
    for onset in [0.4, 1.2, 2.0]:
        tau = t - onset
        active = tau >= 0
        u = tau[active]
        demo[active] += np.exp(-u / 0.12) * (
            0.6 * np.sin(2 * np.pi * 220 * u)
            + 0.25 * np.sin(2 * np.pi * 1400 * u)
            + 0.15 * rng.standard_normal(len(u)))
    demo *= 0.8 / max(np.max(np.abs(demo)), 1e-12)
    AUDIO_PATH = ROOT / "data" / "generated" / "audio_basics_demo.wav"
    AUDIO_PATH.parent.mkdir(parents=True, exist_ok=True)
    sf.write(AUDIO_PATH, demo, demo_sr, subtype="PCM_16")
else:
    AUDIO_PATH = Path(AUDIO_PATH).expanduser()
    if not AUDIO_PATH.is_absolute():
        AUDIO_PATH = ROOT / AUDIO_PATH
if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"WAV bulunamadı: {AUDIO_PATH}")
print("Dosya:", AUDIO_PATH)


## 3. WAV oku ve dinle
`sf.read` dosyayı sayısal diziye çevirir. `always_2d=True` çıktıyı `(örnek, kanal)` biçiminde tutar. Analiz için kanal ortalamasıyla mono kullanıyoruz. Orijinal dosyayı değiştirmiyoruz.
`normalize=False` ses oynatıcısının her kaydı otomatik tepe normalizasyonuyla eşitlemesini engeller; yine de duyulan seviye cihaz ses ayarına bağlıdır.

In [ ]:
info = sf.info(AUDIO_PATH)
audio, sr = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
if len(audio) == 0 or not np.isfinite(audio).all():
    raise ValueError("Dosya boş veya geçersiz örnek içeriyor.")
y = audio.mean(axis=1)
duration = len(y) / sr
peak = float(np.max(np.abs(y)))
rms = float(np.sqrt(np.mean(y.astype(np.float64)**2)))
print(info)
print(f"Kanal: {audio.shape[1]} | Süre: {duration:.3f} s | Nyquist: {sr/2:.0f} Hz")
print(f"Mono peak: {peak:.4f} | RMS: {rms:.4f}")
if peak == 0:
    print("Bu kayıt mono analizde sessiz; spektrumda anlamlı ses içeriği bekleme.")
if peak > 1:
    print("Float örnekler ±1 sınırını aşıyor; oynatmadan önce ölçeği kontrol et.")
else:
    display(Audio(y, rate=sr, normalize=False))


## 4. Waveform — zaman alanı
**Waveform**, genliğin zamanla değişimini gösterir. Yatay eksen saniye, dikey eksen genliktir. Darbelerin nerede başladığını ve sönmesini görürsün; frekans içeriğini doğrudan ayıramazsın.
**Peak** en büyük mutlak genliktir. **RMS** (root mean square), ortalama karesel genliğin kareköküdür: $\sqrt{\frac{1}{N}\sum_n y[n]^2}$. RMS seviye için basit bir göstergedir; algısal loudness veya kalibre edilmiş dB SPL değildir.

In [ ]:
time = np.arange(len(y)) / sr
fig, ax = plt.subplots()
ax.plot(time, y, linewidth=0.6)
ax.set(xlabel="Zaman (s)", ylabel="Genlik", title="Waveform")
ax.set_xlim(0, duration)
plt.tight_layout()
fig.savefig(FIGURES / "waveform.svg", bbox_inches="tight")
plt.show()


## 5. Spectrogram — zaman ve frekans birlikte
**FFT**, bir sinyal parçasını frekans bileşenlerine ayırır. **STFT** (Short-Time Fourier Transform), bunu ses boyunca kaydırılan kısa pencerelerde yapar. **Spectrogram**, STFT büyüklüğünü veya karesini renklerle gösterir. Burada karesini, yani güç temsili kullanıyoruz; fiziksel watt veya kalibre edilmiş güç yoğunluğu değildir.

| Ayar | İşlevi |
|---|---|
| `n_fft` | FFT uzunluğu. Bu notebook'ta pencere uzunluğu da aynı. |
| `window="hann"` | Pencere kenarlarını yumuşatıp spectral leakage'i azaltır. Leakage, sonlu bir parçanın frekans enerjisinin komşu kutulara yayılmasıdır. |
| `hop_length` | Ardışık pencereler arasındaki örnek sayısı. Küçük hop daha sık zaman noktaları verir. |
| Frame | Analiz edilen kısa ses parçası. |
| Bin | FFT'nin bir frekans kutusu. Kutular arası aralık `sr / n_fft`. |
| `center=True` | Kenarlara sıfırlar ekleyerek frame'leri zaman noktalarının üzerine merkezler. |

Uzun pencere yakın frekansları ayırmaya yardım eder ama hızlı olayları zamanda yayar. Kısa pencere bunun tersidir. Bin aralığı, her durumda gerçek ayırt etme çözünürlüğü değildir; pencere tipi ve uzunluğu da önemlidir.

**dB (decibel)** logaritmik bir oran: güç için $10\log_{10}(P/P_{ref})$. `ref=np.max` ile en güçlü hücre 0 dB olur, diğerleri negatiftir. Bu dB SPL değildir. Her grafiğin kendi maksimumuna göre renklenmesi farklı kayıtların mutlak seviyelerini karşılaştırmaya uygun değildir. Sessiz sinyalde 0 dB görünümü ses olduğu anlamına gelmez.

In [ ]:
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 64
print(f"Pencere: {1000*N_FFT/sr:.1f} ms | Hop: {1000*HOP_LENGTH/sr:.1f} ms")
print(f"FFT bin aralığı: {sr/N_FFT:.2f} Hz")
D = librosa.stft(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH,
                 win_length=N_FFT, window="hann", center=True)
power = np.abs(D)**2
power_db = librosa.power_to_db(power, ref=np.max, top_db=80)
fig, ax = plt.subplots()
img = librosa.display.specshow(power_db, sr=sr, hop_length=HOP_LENGTH,
                               x_axis="time", y_axis="linear", ax=ax,
                               vmin=-80, vmax=0)
fig.colorbar(img, ax=ax, label="dB (kendi maksimumuna göre)")
ax.set(title="Power spectrogram", xlabel="Zaman (s)", ylabel="Frekans (Hz)")
plt.tight_layout()
fig.savefig(FIGURES / "spectrogram.svg", bbox_inches="tight")
plt.show()
print("STFT (frekans bin, frame):", D.shape)


## 6. Mel-spectrogram — frekansları algıya yakın gruplamak
**Mel ölçeği**, frekans algısını yaklaşık temsil eden doğrusal olmayan bir ölçektir. Düşük frekanslarda daha sık, yüksek frekanslarda daha geniş aralıklı bantlar kullanılır. İnsan işitmesinin bütün özelliklerini veya loudness'ı ölçmez.
**Mel filter bank**, FFT güç kutularını ağırlıklandırarak mel bantlarında birleştiren filtrelerdir. `n_mels=64` çıktıda 64 bant oluşturur. **Log-mel**, mel gücünün burada dB'ye dönüştürülmüş halidir.

Bu dönüşüm ayrıntıyı sıkıştırır. Mel-spectrogram orijinal waveform değildir ve tek başına faz içermez; ondan sesi yeniden üretmek ayrı bir işlemdir. Model seçimi yapmadan yalnızca temsili inceliyoruz.

In [ ]:
mel_power = librosa.feature.melspectrogram(
    S=power, sr=sr, n_fft=N_FFT, n_mels=N_MELS, fmin=0, fmax=sr/2)
mel_db = librosa.power_to_db(mel_power, ref=np.max, top_db=80)
fig, ax = plt.subplots()
img = librosa.display.specshow(mel_db, sr=sr, hop_length=HOP_LENGTH,
                               x_axis="time", y_axis="mel", fmin=0, fmax=sr/2,
                               ax=ax, vmin=-80, vmax=0)
fig.colorbar(img, ax=ax, label="dB (kendi maksimumuna göre)")
ax.set(title="Log-mel spectrogram", xlabel="Zaman (s)", ylabel="Frekans (Hz, mel ölçeği)")
plt.tight_layout()
fig.savefig(FIGURES / "mel_spectrogram.svg", bbox_inches="tight")
plt.show()
print("Mel (mel bandı, frame):", mel_power.shape)
assert power.shape[0] == N_FFT // 2 + 1
assert mel_power.shape == (N_MELS, power.shape[1])
assert np.isfinite(power_db).all() and np.isfinite(mel_db).all()


## 7. Küçük deney: yalnızca genliği değiştir
Sesin genliğini yarıya indirince waveform küçülür ve RMS yaklaşık 6.02 dB azalır. Frekanslar ve süre değişmez. Ancak her spectrogram kendi maksimumuna göre normalize edilirse iki renkli grafik benzer görünebilir: görselleştirme gerçek seviye değişimini gizleyebilir. Bu, ileride controllability ölçerken dikkat edeceğimiz bir ayrıntıdır.

In [ ]:
if rms > 0:
    quieter = 0.5 * y
    quieter_rms = np.sqrt(np.mean(quieter.astype(np.float64)**2))
    delta_db = 20 * np.log10(quieter_rms / rms)
    print(f"Genlik ×0.5 → RMS oranı: {quieter_rms/rms:.3f}, değişim: {delta_db:.2f} dB")
else:
    print("Sessiz kayıtta RMS oranı tanımsızdır; bu deney atlandı.")


## 8. Sonuçları kaydet
Grafikler `results/figures/01_audio_basics/` içinde SVG olarak kaydedilir.
`summary.csv` ölçümleri, `run.json` dosya hash'i, ayarlar ve paket sürümlerini tutar.
Bunlar bu çalıştırmanın sonuçlarıdır; herhangi bir makalenin deney sonucu değildir.
Aynı yollar sonraki çalıştırmada güncellenir. Başka bir sesi karşılaştırmak için önce mevcut çıktıları farklı bir deney klasörüne kopyala.
Makale bağlantıları: [kod–literatür eşleştirmesi](../papers/notes/code_literature_map.md).

In [ ]:
metrics = {
    "sample_rate_hz": int(sr), "channels": int(audio.shape[1]),
    "samples_mono": len(y), "duration_s": duration,
    "peak_mono": peak, "rms_mono": rms,
    "window_ms": 1000*N_FFT/sr, "hop_ms": 1000*HOP_LENGTH/sr,
    "fft_bin_spacing_hz": sr/N_FFT,
    "stft_frequency_bins": int(power.shape[0]), "frames": int(power.shape[1]),
    "mel_bands": int(mel_power.shape[0]),
    "half_amplitude_rms_ratio": float(quieter_rms/rms) if rms > 0 else None,
    "half_amplitude_delta_db": float(delta_db) if rms > 0 else None,
}
with (TABLES / "summary.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerows(metrics.items())
try:
    source_path = str(AUDIO_PATH.resolve().relative_to(ROOT.resolve()))
except ValueError:
    source_path = AUDIO_PATH.name  # Dış dosyanın özel yerel yolunu kaydetme.
run = {
    "experiment_id": "01_audio_basics",
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "source": source_path,
    "audio_sha256": hashlib.sha256(AUDIO_PATH.read_bytes()).hexdigest(),
    "python": platform.python_version(), "platform": platform.system(),
    "packages": {p: importlib.metadata.version(p) for p in
                 ["numpy", "scipy", "matplotlib", "librosa", "soundfile"]},
    "analysis": {"n_fft": N_FFT, "win_length": N_FFT,
                 "hop_length": HOP_LENGTH, "n_mels": N_MELS,
                 "window": "hann", "center": True, "fmin": 0,
                 "fmax": sr/2, "power_exponent": 2,
                 "db_reference": "per-representation maximum", "top_db": 80},
    "metrics": metrics,
    "scope": "audio representation demo; no trained model or paper reproduction",
}
(EXPERIMENT / "run.json").write_text(
    json.dumps(run, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Kaydedildi:", FIGURES.relative_to(ROOT), TABLES.relative_to(ROOT),
      (EXPERIMENT / "run.json").relative_to(ROOT))


## Bu adımda ne öğrendik?
- WAV bir dosya; waveform zaman alanındaki sayısal sinyalin grafiği.
- Spectrogram sesin hangi anında hangi frekansların güçlü olduğunu gösterir.
- Mel-spectrogram frekansları mel bantlarında gruplar.

**Kendin kontrol et:** Demo darbelerini waveform ve spectrogram'da bul. `N_FFT` değerini 512 ve 2048 yapıp ilgili hücreden aşağısını yeniden çalıştır. Zaman/frekans ayrıntısındaki değişimi gözle. Bu ayarlar analiz ayarlarıdır; henüz generative model'in conditioning parametreleri değildir.

### Kaynaklar
API ayrıntıları için resmi dokümantasyon:
- [librosa STFT](https://librosa.org/doc/latest/generated/librosa.stft.html)
- [librosa mel-spectrogram](https://librosa.org/doc/latest/generated/librosa.feature.melspectrogram.html)
- [librosa power_to_db](https://librosa.org/doc/latest/generated/librosa.power_to_db.html)
- [SoundFile](https://python-soundfile.readthedocs.io/)
